# SIRD Observation Plots

This notebook plots the four SIRD compartments over time using only `Datos/sird_observations.csv`. Vertical boundaries identify the different epidemiological phases, and the annotations report the midpoint gamma value used in each phase.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

DATA_FILE = Path.cwd().parent / "Datos" / "sird_observations.csv"
observations = pd.read_csv(DATA_FILE, parse_dates=["Date"])
observations = observations.sort_values("Date").set_index("Date")
observations.head()

In [ ]:
required_columns = {
    "S", "I", "R", "D", "Epidemiological Phase",
    "Midpoint gamma", "Midpoint beta", "Midpoint delta"
}
missing_columns = required_columns.difference(observations.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

observations[["S", "I", "R", "D"]].describe()

In [ ]:
# Find phase boundaries and midpoint gamma values.
phase_changes = observations.index[
    observations["Epidemiological Phase"]
    .ne(observations["Epidemiological Phase"].shift())
]
phase_intervals = []
for position, start_date in enumerate(phase_changes):
    rows = observations.loc[start_date:]
    if position + 1 < len(phase_changes):
        end_date = observations.index[
            observations.index.get_loc(phase_changes[position + 1]) - 1
        ]
    else:
        end_date = observations.index[-1]
    phase = observations.loc[start_date, "Epidemiological Phase"]
    midpoint_gamma = observations.loc[start_date, "Midpoint gamma"]
    phase_intervals.append((start_date, end_date, phase, midpoint_gamma))

pd.DataFrame(
    phase_intervals,
    columns=["Start", "End", "Phase", "Midpoint gamma"],
)

In [ ]:
compartments = {
    "S": ("Susceptible population", "tab:blue"),
    "I": ("Active infections", "tab:red"),
    "R": ("Recovered population", "tab:green"),
    "D": ("Deaths", "tab:gray"),
}
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
for axis, (column, (title, color)) in zip(axes.flat, compartments.items()):
    axis.plot(observations.index, observations[column], color=color, linewidth=1.8)
    axis.set_title(title)
    axis.set_ylabel("People")
    axis.grid(alpha=0.25)
    axis.ticklabel_format(axis="y", style="plain")

    for start_date, end_date, _, midpoint_gamma in phase_intervals:
        axis.axvline(start_date, color="black", linestyle="--", linewidth=0.8, alpha=0.45)
        midpoint_date = start_date + (end_date - start_date) / 2
        y_position = axis.get_ylim()[1]
        axis.text(
            midpoint_date, y_position * 0.96, f"γ={midpoint_gamma:.4f}",
            ha="center", va="top", fontsize=8, rotation=90, alpha=0.8
        )

axes[1, 0].set_xlabel("Date")
axes[1, 1].set_xlabel("Date")
axes[0, 0].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
for axis in axes.flat:
    axis.tick_params(axis="x", rotation=45)

fig.suptitle("SIRD Compartments and Midpoint Gamma by Epidemiological Phase", fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
# Plot the epidemiological phases separately for a clearer gamma comparison.
phases = observations[["Epidemiological Phase", "Midpoint gamma"]].drop_duplicates()
fig, axis = plt.subplots(figsize=(14, 4))
for position, (_, row) in enumerate(phases.iterrows(), start=1):
    axis.scatter(position, row["Midpoint gamma"], s=80, color="tab:purple")
    axis.annotate(
        row["Epidemiological Phase"],
        (position, row["Midpoint gamma"]),
        xytext=(0, 12), textcoords="offset points",
        ha="center", fontsize=9, rotation=30
    )
axis.set_xticks(range(1, len(phases) + 1))
axis.set_xticklabels([row["Epidemiological Phase"] for _, row in phases.iterrows()])
axis.set_ylabel("Midpoint gamma (day$^{-1}$)")
axis.set_title("Midpoint Gamma by Epidemiological Phase")
axis.grid(alpha=0.25)
axis.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()